In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
import pandas as pd
import numpy as np
import csv

def build_crime_feature_file(
    file_path,
    output_path=None,
    time_freq="h",
    use_full_day_grid=True
):
    """
    Build a processed feature table from raw incident-level crime data.

    Input:
        Raw CSV with at least:
        - Date
        - Community Area

    Output:
        One row per (Community Area, hour_slot), with engineered features.
    """

    # 1. Load only needed columns safely
    try:
        df = pd.read_csv(
            file_path,
            usecols=["Date", "Community Area"],
            engine="python",
            on_bad_lines="skip",
            encoding_errors="replace"
        )
    except Exception as e:
        print("Normal CSV read failed. Trying fallback reader...")
        print(e)

        df = pd.read_csv(
            file_path,
            usecols=["Date", "Community Area"],
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip",
            encoding_errors="replace"
        )

    # 2. Parse and clean
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")

    df = df.dropna(subset=["Date", "Community Area"]).copy()
    df["Community Area"] = df["Community Area"].astype(int)

    # 3. Create time slot
    df["hour_slot"] = df["Date"].dt.floor(time_freq)

    # 4. Aggregate to Community Area + hour
    crime_counts = (
        df.groupby(["Community Area", "hour_slot"])
          .size()
          .reset_index(name="crime_count")
    )

    crime_counts["crime_occurrence"] = (
        crime_counts["crime_count"] > 0
    ).astype(int)

    # 5. Create full grid
    all_areas = sorted(df["Community Area"].unique())

    if use_full_day_grid:
        start_day = df["Date"].dt.floor("D").min()
        end_day = df["Date"].dt.floor("D").max()

        all_hours = pd.date_range(
            start=start_day,
            end=end_day + pd.Timedelta(hours=23),
            freq=time_freq
        )
    else:
        all_hours = pd.date_range(
            start=df["hour_slot"].min(),
            end=df["hour_slot"].max(),
            freq=time_freq
        )

    full_grid = pd.MultiIndex.from_product(
        [all_areas, all_hours],
        names=["Community Area", "hour_slot"]
    ).to_frame(index=False)

    # 6. Merge counts into full grid
    data = full_grid.merge(
        crime_counts,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    data["crime_count"] = data["crime_count"].fillna(0)
    data["crime_occurrence"] = data["crime_occurrence"].fillna(0).astype(int)

    # 7. Time features
    data["hour"] = data["hour_slot"].dt.hour
    data["day_of_week"] = data["hour_slot"].dt.dayofweek
    data["month"] = data["hour_slot"].dt.month
    data["day"] = data["hour_slot"].dt.day
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

    # 8. Cyclical hour encoding
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    # 9. Sort before lag features
    data = data.sort_values(
        ["Community Area", "hour_slot"]
    ).reset_index(drop=True)

    # 10. Lag features based on past crime_count
    grouped = data.groupby("Community Area")["crime_count"]

    data["lag_1h"] = grouped.shift(1)
    data["lag_2h"] = grouped.shift(2)
    data["lag_3h"] = grouped.shift(3)
    data["lag_24h"] = grouped.shift(24)

    # 11. Rolling features using only past values
    data["rolling_3h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(3).mean()
    )

    data["rolling_6h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(6).mean()
    )

    data["rolling_12h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(12).mean()
    )

    data["rolling_24h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(24).mean()
    )

    # 12. Fill missing lag/rolling values
    lag_roll_cols = [
        "lag_1h", "lag_2h", "lag_3h", "lag_24h",
        "rolling_3h_mean", "rolling_6h_mean",
        "rolling_12h_mean", "rolling_24h_mean"
    ]

    data[lag_roll_cols] = data[lag_roll_cols].fillna(0)

    # 13. Type cleanup
    int_cols = [
        "Community Area",
        "crime_occurrence",
        "hour",
        "day_of_week",
        "month",
        "day",
        "is_weekend"
    ]

    for col in int_cols:
        data[col] = data[col].astype(int)

    # 14. Save if requested
    if output_path is not None:
        data.to_csv(output_path, index=False)

    return data

In [ ]:
# Define the path to your file in Google Drive
file_path = "/content/Chicago_Crimes_SEIS764.csv"

# Call the function to build the features
processed_df = build_crime_feature_file(
    file_path=file_path
)

In [ ]:
processed_df.columns

In [ ]:
# target column
target_col = "crime_occurrence"

# feature columns
feature_cols = [
    "Community Area",
    "day_of_week",
    "month",
    "day",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "rolling_3h_mean",
    "rolling_6h_mean",
    "rolling_12h_mean",
    "rolling_24h_mean"
]

# keep only required columns
model_df = processed_df[feature_cols + [target_col]].copy()



In [ ]:
model_df.head()

Section 1: Filter Only Actual Crime Records

In [ ]:
crime_only_df = processed_df[
    processed_df["crime_occurrence"] == 1
].copy()

total_crime_records = len(crime_only_df)

print("Total actual crime records used for EDA:", total_crime_records)

print("\nCrime occurrence values after filtering:")
print(crime_only_df["crime_occurrence"].value_counts())

crime_only_df.head()

Section 2: Community Area Share of Actual Crime Records

In [ ]:
area_counts = (
    crime_only_df["Community Area"]
    .value_counts()
    .sort_values(ascending=False)
)

area_summary = pd.DataFrame({
    "Community_Area": area_counts.index,
    "Crime_1_Count": area_counts.values,
    "Percentage_of_Crime_Records": (area_counts.values / total_crime_records) * 100
})

display(area_summary)

This section counts only crime_occurrence = 1 records by Community Area. Crime_1_Count shows how many actual crime records occurred in each area. Percentage_of_Crime_Records shows each area's share out of all actual crime records.


Section 3: Top 5 Community Areas Where Crime Happened Most

In [ ]:
top5_area_share = area_summary.head(5)

plt.figure(figsize=(8,5))

plt.hlines(
    y=top5_area_share["Community_Area"].astype(str),
    xmin=0,
    xmax=top5_area_share["Percentage_of_Crime_Records"]
)

plt.plot(
    top5_area_share["Percentage_of_Crime_Records"],
    top5_area_share["Community_Area"].astype(str),
    "o"
)

plt.title("Top 5 Community Areas with Highest Share of Actual Crime Records")
plt.xlabel("Percentage of Crime Records (%)")
plt.ylabel("Community Area")
plt.grid(True)
plt.show()

display(top5_area_share)

display(top5_area_share)

Community Area 25 has the highest share of actual crime records, followed by Community Areas 8, 28, 43, and 32. These percentages are calculated only from records where crime_occurrence = 1.

Section 4: Bottom 5 Community Areas Where Crime Happened Least

In [ ]:
bottom5_area_share = area_summary.tail(5)

plt.figure(figsize=(8,5))

plt.hlines(
    y=bottom5_area_share["Community_Area"].astype(str),
    xmin=0,
    xmax=bottom5_area_share["Percentage_of_Crime_Records"]
)

plt.plot(
    bottom5_area_share["Percentage_of_Crime_Records"],
    bottom5_area_share["Community_Area"].astype(str),
    "o"
)

plt.title("Bottom 5 Community Areas with Lowest Share of Actual Crime Records")
plt.xlabel("Percentage of Crime Records (%)")
plt.ylabel("Community Area")
plt.grid(True)
plt.show()

display(bottom5_area_share)

Section 5: Hourly Distribution of Actual Crime Records

In [ ]:
hour_counts = crime_only_df["hour"].value_counts().sort_index()

hour_summary = pd.DataFrame({
    "Hour": hour_counts.index,
    "Crime_1_Count": hour_counts.values,
    "Percentage_of_Crime_Records": (hour_counts.values / total_crime_records) * 100
})

plt.figure(figsize=(10,5))

plt.plot(
    hour_summary["Hour"],
    hour_summary["Percentage_of_Crime_Records"],
    marker="o"
)

plt.title("Percentage of Actual Crime Records by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Percentage of Crime Records (%)")
plt.xticks(range(0,24))
plt.grid(True)
plt.show()

display(hour_summary)

print("Hour with highest share of actual crime records:")
display(hour_summary.sort_values("Percentage_of_Crime_Records", ascending=False).head(1))

print("Hour with lowest share of actual crime records:")
display(hour_summary.sort_values("Percentage_of_Crime_Records", ascending=True).head(1))

This section shows when actual crimes happened across the day. The percentage is calculated only from crime_occurrence = 1 records.

Section 6: Difference from Average Crime Share by Hour

In [ ]:
average_hour_share = hour_summary["Percentage_of_Crime_Records"].mean()

hour_summary["Difference_From_Average"] = (
    hour_summary["Percentage_of_Crime_Records"] - average_hour_share
)

plt.figure(figsize=(10,5))

plt.stem(
    hour_summary["Hour"],
    hour_summary["Difference_From_Average"]
)

plt.axhline(0, linestyle="--")

plt.title("Difference from Average Crime Share by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Difference from Average (%)")
plt.xticks(range(0,24))
plt.grid(True)
plt.show()

display(hour_summary)

Positive values mean that hour has a higher-than-average share of actual crime records. Negative values mean that hour has a lower-than-average share.

Section 7: Day-of-Week Distribution of Actual Crime Records

In [ ]:
day_labels = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

day_counts = crime_only_df["day_of_week"].value_counts().sort_index()

day_summary = pd.DataFrame({
    "day_of_week": day_counts.index,
    "Day_Name": [day_labels[i] for i in day_counts.index],
    "Crime_1_Count": day_counts.values,
    "Percentage_of_Crime_Records": (day_counts.values / total_crime_records) * 100
})

x = np.arange(len(day_summary))

plt.figure(figsize=(9,5))

plt.plot(
    x,
    day_summary["Percentage_of_Crime_Records"],
    marker="o"
)

plt.fill_between(
    x,
    day_summary["Percentage_of_Crime_Records"],
    alpha=0.3
)

plt.title("Percentage of Actual Crime Records by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Percentage of Crime Records (%)")
plt.xticks(x, day_summary["Day_Name"], rotation=30)
plt.grid(True)
plt.show()

display(day_summary)

print("Day with highest share of actual crime records:")
display(day_summary.sort_values("Percentage_of_Crime_Records", ascending=False).head(1))

print("Day with lowest share of actual crime records:")
display(day_summary.sort_values("Percentage_of_Crime_Records", ascending=True).head(1))

The bottom 5 lowest crime occurrence hours are 5 AM, 6 AM, 4 AM, 3 AM, and 7 AM. The lowest crime occurrence rate is at 5 AM, with a rate of about 11.47%. This suggests that early morning hours have lower recorded crime activity compared with other times of the day.

Section 8: Monthly Distribution of Actual Crime Records

In [ ]:


# Close old/blank plots
plt.close("all")

# Use only actual crime records
crime_only_df = processed_df[
    processed_df["crime_occurrence"] == 1
].copy()

total_crime_records = len(crime_only_df)

# Count actual crime records by month
month_counts = crime_only_df["month"].value_counts().sort_index()

month_summary = pd.DataFrame({
    "Month": month_counts.index,
    "Crime_1_Count": month_counts.values,
    "Percentage_of_Crime_Records": (month_counts.values / total_crime_records) * 100
})

display(month_summary)

plt.figure(figsize=(8,5))

plt.scatter(
    month_summary["Month"],
    month_summary["Percentage_of_Crime_Records"],
    s=month_summary["Percentage_of_Crime_Records"] * 30
)

for x, y in zip(month_summary["Month"], month_summary["Percentage_of_Crime_Records"]):
    plt.text(x, y + 0.2, f"{y:.2f}%", ha="center")

plt.title("Percentage of Actual Crime Records by Month")
plt.xlabel("Month")
plt.ylabel("Percentage of Crime Records (%)")

plt.xticks(range(1, 13))

# Important: make y-axis large enough
plt.ylim(0, month_summary["Percentage_of_Crime_Records"].max() + 2)

plt.grid(True)
plt.show()

print("Month with highest share of actual crime records:")
display(month_summary.sort_values("Percentage_of_Crime_Records", ascending=False).head(1))

print("Month with lowest share of actual crime records:")
display(month_summary.sort_values("Percentage_of_Crime_Records", ascending=True).head(1))

This section uses only records where crime_occurrence = 1. The percentage shows how actual crime records are distributed by month. No-crime records are excluded from the calculation.

Section 9: Day-Hour Heatmap of Actual Crime Records


In [ ]:
day_hour_counts = crime_only_df.pivot_table(
    values="crime_occurrence",
    index="day_of_week",
    columns="hour",
    aggfunc="count",
    fill_value=0
)

day_hour_percent = (day_hour_counts / total_crime_records) * 100

plt.figure(figsize=(14,6))

plt.imshow(day_hour_percent, aspect="auto")
plt.colorbar(label="Percentage of Actual Crime Records (%)")

plt.title("Actual Crime Record Share by Day of Week and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.xticks(range(24), range(24))
plt.yticks(
    range(7),
    ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
)

plt.show()

This heatmap shows how actual crime records are distributed across both day of week and hour of day. The values are percentages of total crime_occurrence = 1 records.


Section 10: Lag Feature Distribution for Actual Crime Records

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(
    [
        crime_only_df["lag_1h"],
        crime_only_df["lag_2h"],
        crime_only_df["lag_3h"],
        crime_only_df["lag_24h"]
    ],
    labels=["lag_1h", "lag_2h", "lag_3h", "lag_24h"]
)

plt.title("Lag Feature Distribution for Actual Crime Records")
plt.xlabel("Lag Feature")
plt.ylabel("Previous Crime Count")
plt.grid(True)
plt.show()

Section 11: Rolling Feature Pattern for Actual Crime Records

In [ ]:
rolling_means = crime_only_df[
    ["rolling_3h_mean", "rolling_6h_mean", "rolling_12h_mean", "rolling_24h_mean"]
].mean()

plt.figure(figsize=(9,5))
plt.plot(rolling_means.index, rolling_means.values, marker="o")

plt.title("Average Rolling Crime Features for Actual Crime Records")
plt.xlabel("Rolling Feature")
plt.ylabel("Average Rolling Mean")
plt.xticks(rotation=30)
plt.grid(True)
plt.show()

display(rolling_means)

In [ ]:
# ============================================================
# EDA: No-Crime Records Only
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep only rows where crime did NOT happen
no_crime_df = processed_df[
    processed_df["crime_occurrence"] == 0
].copy()

total_no_crime_records = len(no_crime_df)

print("Total no-crime records used for EDA:", total_no_crime_records)

print("\nCrime occurrence values after filtering:")
print(no_crime_df["crime_occurrence"].value_counts())

no_crime_df.head()

Section 2: Community Area Share of No-Crime Records

In [ ]:
area_zero_counts = (
    no_crime_df["Community Area"]
    .value_counts()
    .sort_values(ascending=False)
)

area_zero_summary = pd.DataFrame({
    "Community_Area": area_zero_counts.index,
    "No_Crime_0_Count": area_zero_counts.values,
    "Percentage_of_No_Crime_Records": (
        area_zero_counts.values / total_no_crime_records
    ) * 100
})

display(area_zero_summary)

Section 3: Top 5 Community Areas with Most No-Crime Records

In [ ]:


top5_no_crime_area = area_zero_summary.head(5)

plt.figure(figsize=(8,5))

plt.hlines(
    y=top5_no_crime_area["Community_Area"].astype(str),
    xmin=0,
    xmax=top5_no_crime_area["Percentage_of_No_Crime_Records"]
)

plt.plot(
    top5_no_crime_area["Percentage_of_No_Crime_Records"],
    top5_no_crime_area["Community_Area"].astype(str),
    "o"
)

plt.title("Top 5 Community Areas with Highest Share of No-Crime Records")
plt.xlabel("Percentage of No-Crime Records (%)")
plt.ylabel("Community Area")
plt.grid(True)
plt.show()

display(top5_no_crime_area)

Section 4: Bottom 5 Community Areas with Fewest No-Crime Records

In [ ]:
bottom5_no_crime_area = area_zero_summary.tail(5)

plt.figure(figsize=(8,5))

plt.hlines(
    y=bottom5_no_crime_area["Community_Area"].astype(str),
    xmin=0,
    xmax=bottom5_no_crime_area["Percentage_of_No_Crime_Records"]
)

plt.plot(
    bottom5_no_crime_area["Percentage_of_No_Crime_Records"],
    bottom5_no_crime_area["Community_Area"].astype(str),
    "o"
)

plt.title("Bottom 5 Community Areas with Lowest Share of No-Crime Records")
plt.xlabel("Percentage of No-Crime Records (%)")
plt.ylabel("Community Area")
plt.grid(True)
plt.show()

display(bottom5_no_crime_area)

Section 5: Hourly Distribution of No-Crime Records

In [ ]:
hour_zero_counts = no_crime_df["hour"].value_counts().sort_index()

hour_zero_summary = pd.DataFrame({
    "Hour": hour_zero_counts.index,
    "No_Crime_0_Count": hour_zero_counts.values,
    "Percentage_of_No_Crime_Records": (
        hour_zero_counts.values / total_no_crime_records
    ) * 100
})

plt.figure(figsize=(10,5))

plt.plot(
    hour_zero_summary["Hour"],
    hour_zero_summary["Percentage_of_No_Crime_Records"],
    marker="o"
)

plt.title("Percentage of No-Crime Records by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Percentage of No-Crime Records (%)")
plt.xticks(range(0,24))
plt.grid(True)
plt.show()

display(hour_zero_summary)

print("Hour with highest share of no-crime records:")
display(hour_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=False
).head(1))

print("Hour with lowest share of no-crime records:")
display(hour_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=True
).head(1))

Section 6: Difference from Average No-Crime Share by Hour

In [ ]:


average_hour_zero_share = hour_zero_summary[
    "Percentage_of_No_Crime_Records"
].mean()

hour_zero_summary["Difference_From_Average"] = (
    hour_zero_summary["Percentage_of_No_Crime_Records"] -
    average_hour_zero_share
)

plt.figure(figsize=(10,5))

plt.stem(
    hour_zero_summary["Hour"],
    hour_zero_summary["Difference_From_Average"]
)

plt.axhline(0, linestyle="--")

plt.title("Difference from Average No-Crime Share by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Difference from Average (%)")
plt.xticks(range(0,24))
plt.grid(True)
plt.show()

display(hour_zero_summary)

Section 7: Day-of-Week Distribution of No-Crime Records

In [ ]:
day_labels = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

day_zero_counts = no_crime_df["day_of_week"].value_counts().sort_index()

day_zero_summary = pd.DataFrame({
    "day_of_week": day_zero_counts.index,
    "Day_Name": [day_labels[i] for i in day_zero_counts.index],
    "No_Crime_0_Count": day_zero_counts.values,
    "Percentage_of_No_Crime_Records": (
        day_zero_counts.values / total_no_crime_records
    ) * 100
})

x = np.arange(len(day_zero_summary))

plt.figure(figsize=(9,5))

plt.plot(
    x,
    day_zero_summary["Percentage_of_No_Crime_Records"],
    marker="o"
)

plt.fill_between(
    x,
    day_zero_summary["Percentage_of_No_Crime_Records"],
    alpha=0.3
)

plt.title("Percentage of No-Crime Records by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Percentage of No-Crime Records (%)")
plt.xticks(x, day_zero_summary["Day_Name"], rotation=30)
plt.grid(True)
plt.show()

display(day_zero_summary)

print("Day with highest share of no-crime records:")
display(day_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=False
).head(1))

print("Day with lowest share of no-crime records:")
display(day_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=True
).head(1))

Section 8: Monthly Distribution of No-Crime Records

In [ ]:
month_zero_counts = no_crime_df["month"].value_counts().sort_index()

month_zero_summary = pd.DataFrame({
    "Month": month_zero_counts.index,
    "No_Crime_0_Count": month_zero_counts.values,
    "Percentage_of_No_Crime_Records": (
        month_zero_counts.values / total_no_crime_records
    ) * 100
})

plt.figure(figsize=(8,5))

plt.scatter(
    month_zero_summary["Month"],
    month_zero_summary["Percentage_of_No_Crime_Records"],
    s=month_zero_summary["Percentage_of_No_Crime_Records"] * 30
)

for x_val, y_val in zip(
    month_zero_summary["Month"],
    month_zero_summary["Percentage_of_No_Crime_Records"]
):
    plt.text(x_val, y_val + 0.2, f"{y_val:.2f}%", ha="center")

plt.title("Percentage of No-Crime Records by Month")
plt.xlabel("Month")
plt.ylabel("Percentage of No-Crime Records (%)")
plt.xticks(range(1,13))
plt.ylim(
    0,
    month_zero_summary["Percentage_of_No_Crime_Records"].max() + 2
)
plt.grid(True)
plt.show()

display(month_zero_summary)

print("Month with highest share of no-crime records:")
display(month_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=False
).head(1))

print("Month with lowest share of no-crime records:")
display(month_zero_summary.sort_values(
    "Percentage_of_No_Crime_Records",
    ascending=True
).head(1))

Section 9: Day-Hour Heatmap of No-Crime Records

In [ ]:
day_hour_zero_counts = no_crime_df.pivot_table(
    values="crime_occurrence",
    index="day_of_week",
    columns="hour",
    aggfunc="count",
    fill_value=0
)

day_hour_zero_percent = (
    day_hour_zero_counts / total_no_crime_records
) * 100

plt.figure(figsize=(14,6))

plt.imshow(day_hour_zero_percent, aspect="auto")
plt.colorbar(label="Percentage of No-Crime Records (%)")

plt.title("No-Crime Record Share by Day of Week and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.xticks(range(24), range(24))
plt.yticks(
    range(7),
    ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
)

plt.show()

In this EDA, I looked at crime_occurrence in two separate ways: records where crime_occurrence = 1 and records where crime_occurrence = 0.

For crime_occurrence = 1, I analyzed only the rows where a crime actually happened. This helped me understand where and when crime records are concentrated. I checked Community Area, hour of day, day of week, month, day-hour patterns, lag features, and rolling features. The results showed that actual crime records are not evenly spread across all areas or time periods. Some Community Areas had a higher share of crime records than others. Some hours and days also had more crime records compared with the average. This means location and time-based features are useful for understanding crime occurrence.

For crime_occurrence = 0, I analyzed only the rows where no crime was recorded. These records represent Community Area-hour slots where no crime happened. This helped me understand the opposite pattern: where and when no-crime records are more common. Some Community Areas and time periods had a higher share of no-crime records, while others had fewer no-crime records.

By comparing both parts, I observed that crime and no-crime records follow different patterns. Crime records are concentrated more in certain Community Areas and time windows, while no-crime records are more common in other areas and times. This shows that the target variable is not random. Features such as Community Area, hour, day of week, month, lag values, and rolling averages can help the model learn the difference between crime and no-crime situations.

Overall, the EDA shows that crime_occurrence is influenced by both location and time. The dataset also has many no-crime records compared with crime records, so the target is imbalanced. Because of this, model evaluation should not depend only on accuracy. Metrics such as precision, recall, F1-score, and confusion matrix will be important.